In [67]:
import os
import calendar
from datetime import date
from pathlib import Path
import pandas as pd
import numpy as np
import clickhouse_connect
from dotenv import load_dotenv

### Configuration & Setup

In [68]:
load_dotenv()  # Load secrets from a .env file securely

YEAR = 2026
MONTH = 5  # Change this to process different months

BASE_DIR = Path("/Volumes/E$/CEIR/Clean Dumps")  # Update this to your actual base directory
FAKE_DUMP_PATH    = BASE_DIR / "Fake"    / f"fake_{YEAR}-{MONTH:02d}.parquet"
GENUINE_DUMP_PATH = BASE_DIR / "Genuine" / f"genuine_{YEAR}-{MONTH:02d}.parquet"

TIME_COL = "imei_first_seen"   # Timestamp column driving the month filter
IMEI_COL = None   # IMEI dedup done upstream; set to "imei" only if unique counts are needed

# Optional: run OPTIMIZE TABLE ... FINAL after each insert so
# ReplacingMergeTree collapses re-run duplicates immediately
OPTIMIZE_AFTER_INSERT = True

# --- CLICKHOUSE CONNECTION ---
# Make sure your .env file uses these CH_ variables
DB = dict(
    host=os.environ.get("CH_HOST", "localhost"),
    port=int(os.environ.get("CH_PORT", 8123)),
    database=os.environ.get("CH_DATABASE", "default"),
    username=os.environ.get("CH_USER", "default"),
    password=os.environ.get("CH_PASSWORD", ""),
)

### Helper Functions

In [69]:
def month_end_period(year: int, month: int) -> date:
    """Returns the exact last day of the given month."""
    last_day = calendar.monthrange(year, month)[1]
    return date(year, month, last_day)

In [70]:
# FEATURE PREP — derive time parts + age brackets from raw data
# ============================================================
def prep_time_and_age(df: pd.DataFrame, time_col: str = TIME_COL) -> pd.DataFrame:
    # Ensure the timestamp column is real datetime before using .dt
    df[time_col] = pd.to_datetime(df[time_col], errors="coerce")

    # Time dimensions used across the monthly reports
    df["year_month"] = df[time_col].dt.to_period("M")
    df["hour"]       = df[time_col].dt.hour
    df["dow"]        = df[time_col].dt.day_name()

    # Age -> ordered bracket. right=True means each bin is (low, high];
    # include_lowest=True so an exact age of 0 still lands in "<18".
    bins   = [0, 17, 30, 40, 50, 60, 70, 100, 120]
    labels = ["<18", "18-30", "31-40", "41-50", "51-60", "61-70", "71-100", "100-120"]
    numeric_age = pd.to_numeric(df["age"], errors="coerce")
    age_bracket = pd.cut(numeric_age, bins=bins, labels=labels,
                         right=True, include_lowest=True)

    # Missing / out-of-range ages -> "unknown" (no age data on file)
    age_bracket = age_bracket.astype("object").fillna("unknown")
    df["age_bracket"] = pd.Categorical(age_bracket,
                                       categories=labels + ["unknown"],
                                       ordered=True)
    return df

In [71]:
def add_time_dimensions(out: pd.DataFrame) -> pd.DataFrame:
    """Expands the year_month Period column into period / year / month
    and drops the raw Period (ClickHouse cannot ingest pandas Periods)."""
    out["year"]   = out["year_month"].dt.year.astype(int)
    out["month"]  = out["year_month"].dt.month.astype(int)
    out["period"] = [month_end_period(y, m) for y, m in zip(out["year"], out["month"])]
    return out.drop(columns=["year_month"])

### Data Aggregation Builders

In [72]:
def build_by_dimension(
    df: pd.DataFrame,
    status: str,
    dim_col: str,
    *,
    fill: str | None = None,
    dropna: bool = True,
) -> pd.DataFrame:
    """Long-format equivalent of report_by_month(): device counts per
    year_month x dimension, tagged with status ('fake' / 'genuine')."""
    series = df[dim_col]
    # Fill nulls when a label is given (cast to object first so it works
    # even if the column is categorical, e.g. mno/gender/age_bracket)
    if fill is not None:
        series = series.astype("object").fillna(fill)

    out = (
        df.assign(**{dim_col: series})
          .groupby(["year_month", dim_col], observed=True, dropna=dropna)
          .size()
          .reset_index(name="imei_count")
    )

    out = add_time_dimensions(out)
    out.insert(0, "status", status)
    out[dim_col] = out[dim_col].astype(str)   # Categoricals/NaN -> plain strings

    cols_order = ["period", "year", "month", "status", dim_col, "imei_count"]
    return out[cols_order]

In [73]:
def build_month_summ(df: pd.DataFrame, status: str) -> pd.DataFrame:
    """High-level monthly snapshot per status — the 'TOTAL' margin of the
    crosstab reports (IMEI counts per month)."""
    out = (
        df.groupby("year_month", observed=True)
          .agg(imei_count=("year_month", "size"))
          .reset_index()
    )

    out = add_time_dimensions(out)
    out.insert(0, "status", status)

    cols_order = ["period", "year", "month", "status", "imei_count"]
    return out[cols_order]


In [74]:
def build_combined(builder, *args, **kwargs) -> pd.DataFrame:
    """Runs a builder for both fake and genuine frames and stacks the result
    so each table carries a status dimension instead of being duplicated."""
    return pd.concat(
        [
            builder(fake_df, "fake", *args, **kwargs),
            builder(genuine_df, "genuine", *args, **kwargs),
        ],
        ignore_index=True,
    )

### Execution Pipeline

In [75]:
if __name__ == "__main__":

    # 1. Load fake and genuine IMEI datasets
    print("Loading fake and genuine Parquet dumps...")
    fake_df    = pd.read_parquet(FAKE_DUMP_PATH)
    genuine_df = pd.read_parquet(GENUINE_DUMP_PATH)

    # Standardize uppercase column to match database schema
    for frame in (fake_df, genuine_df):
        if "MNO" in frame.columns:
            frame.rename(columns={"MNO": "mno"}, inplace=True)

    # 2. Memory optimization: Convert low-cardinality strings to categories
    print("Optimizing memory with categorical casting...")
    for frame in (fake_df, genuine_df):
        for c in ("mno", "gender", "district", "country"):
            if c in frame.columns:
                frame[c] = frame[c].astype("category")

    # 3. Feature prep: time parts + age brackets
    print("Deriving time dimensions and age brackets...")
    fake_df    = prep_time_and_age(fake_df)
    genuine_df = prep_time_and_age(genuine_df)

Loading fake and genuine Parquet dumps...
Optimizing memory with categorical casting...
Deriving time dimensions and age brackets...


In [76]:
    # 4. Define the processing manifest
    PUBLISH = [
        {
            "database": "ceir",
            "table": "imei_month_summ",
            "build": lambda: build_combined(build_month_summ),
        },
        {
            "database": "ceir",
            "table": "imei_by_mno",
            "build": lambda: build_combined(build_by_dimension, "mno", fill="unknown"),
        },
        {
            "database": "ceir",
            "table": "imei_by_gender",
            "build": lambda: build_combined(build_by_dimension, "gender", fill="unknown"),
        },
        {
            "database": "ceir",
            "table": "imei_by_age",
            "build": lambda: build_combined(build_by_dimension, "age_bracket", dropna=False),
        },
        {
            "database": "ceir",
            "table": "imei_by_district",
            "build": lambda: build_combined(build_by_dimension, "district"),
        },
        {
            "database": "ceir",
            "table": "imei_by_country",
            "build": lambda: build_combined(build_by_dimension, "country", fill="unknown"),
        },
    ]

In [77]:
    # 5. Execute database transactions
    print("\nConnecting to ClickHouse database...")
    client = clickhouse_connect.get_client(**DB)

    try:
        for item in PUBLISH:
            print(f"Building data for {item['database']}.{item['table']}...")
            df_out = item["build"]()
            df_out["updated_at"] = pd.Timestamp.now()

            print(f"Inserting into {item['database']}.{item['table']} ({len(df_out):,} rows)...")
            client.insert_df(
                table=item["table"],
                database=item["database"],
                df=df_out,
            )

            # ReplacingMergeTree dedups lazily; force the collapse so
            # re-runs of the same months don't show duplicate rows
            if OPTIMIZE_AFTER_INSERT:
                client.command(
                    f"OPTIMIZE TABLE {item['database']}.{item['table']} FINAL"
                )

        print("\n✅ Pipeline completed successfully. All data committed.")

    except Exception as e:
        print(f"\n❌ Pipeline failed. Error: {e}")
        raise

    finally:
        client.close()
        print("Database connection closed.")


Connecting to ClickHouse database...
Building data for ceir.imei_month_summ...
Inserting into ceir.imei_month_summ (2 rows)...
Building data for ceir.imei_by_mno...
Inserting into ceir.imei_by_mno (7 rows)...
Building data for ceir.imei_by_gender...
Inserting into ceir.imei_by_gender (8 rows)...
Building data for ceir.imei_by_age...
Inserting into ceir.imei_by_age (16 rows)...
Building data for ceir.imei_by_district...
Inserting into ceir.imei_by_district (272 rows)...
Building data for ceir.imei_by_country...
Inserting into ceir.imei_by_country (139 rows)...

✅ Pipeline completed successfully. All data committed.
Database connection closed.
